# 1. Metodologia de Construção
- A construção deste book seguiu o princípio de Corte Temporal Reversivo para garantir a integridade do modelo e evitar o Data Leakage (vazamento de - dados).

- Ponto de Corte: Para cada registro, foram considerados apenas os eventos de recarga ocorridos em data estritamente inferior à data da SAFRA (dia 01 do mês de referência).

- Granularidade: Os dados transacionais originais (N linhas por CPF) foram agregados para a granularidade de ID_UNICO (1 linha por CPF/Safra).

- Tratamento de Plataformas: Utilizou-se uma tabela de dimensão para categorizar o COD_PLATAFORMA_ATU, permitindo identificar o comportamento de consumo por perfil de produto (Pré, Controle, Pós).

# 2. Dicionário de Variáveis e Relevância Estratégica

| Variável | Lógica de Construção | Relevância para o Risco de Crédito
| :--- | :--- | :--- |
| **`REC_TOTAL_VALOR_HIST`** | Soma de todos os valores de recarga (`VAL_CREDITO_INSERIDO`) no período válido. |Indica o LTV (Lifetime Value) e a capacidade financeira histórica do cliente.
| **`REC_QTD_RECARGAS_HIST`** | Contagem total de eventos de recarga. | Mede a estabilidade. Clientes com recargas frequentes podem demonstram maior engajamento e previsibilidade.
| **`REC_MEDIA_VALOR_HIST`** | Média aritmética dos valores inseridos, arredondada para 2 casas decimais. | Define o ticket médio. Ajuda a segmentar o poder aquisitivo.
| **`REC_MAX_VALOR_HISTT`** | Valor máximo já recarregado em uma única transação. | Indica o teto de desembolso esporádico do cliente.
| **`REC_DIAS_DESDE_ULTIMA`** | Diferença em dias entre a data da Safra e a `DATA_ULTIMA_RECARGA`. | Recência: Possibilidade de ser um preditor forte. Com a hipótese de quanto maior o tempo sem atividade, maior o risco inadimplência.
| **`QTD_PRE_PAGO` / `CONTROLE` / `POS_PAGO`** | Soma binária (Flag 0/1) baseada no de-para da tabela de dimensão de plataformas. | Além de identificar o Plano, permite diferenciar "consumo zero" por falta de uso de "consumo zero" por característica do plano (ex: Controle/Pós).
| **`QTD_SOS`** | Soma da FLAG_SOS convertida em inteiro. | Indica a frequência de uso de Crédito de Emergência. O uso excessivo pode sinalizar fragilidade financeira momentânea.
| **`TOTAL_VALOR_SOS`** | Soma acumulada dos valores de crédito emergencial contratados. | Quantifica a dependência do cliente em relação a microempréstimos da operadora.
| **`MEDIA_SOS`** | Valor médio das solicitações de SOS, ignorando períodos sem uso. | Identifica o comportamento padrão em situações de falta de saldo.


# 3. Considerações Técnicas sobre a Integração (Inner Join)
A opção pelo Inner Join entre a df_gold e o df_book_recarga foi estratégica para esta fase de Baseline. Esta decisão garante que:

* 1 - Qualidade da Informação: O modelo será treinado apenas com indivíduos que possuem histórico de atividade mensurável, reduzindo o ruído causado por CPFs inativos ou sem informações de consumo.

* 2 -Saneamento de Tipos: Todas as variáveis monetárias e de média foram padronizadas para duas casas decimais, assegurando que o algoritmo não interprete variações infinitesimais de ponto flutuante como informação relevante.

* 3 - Variáveis Sentinela: Manteve-se a coerência com as flags de missing tratadas anteriormente, permitindo que o modelo diferencie a ausência de informação do valor zero real.

# 4. Próximos Passos
Para a continuidade do projeto e refinamento da ABT, as seguintes frentes de trabalho serão priorizadas:

* Análise de Tendência e Momento de Consumo: Implementação de variáveis de variação temporal (ex: média de recargas nos últimos 3 meses vs. últimos 6 meses). O objetivo é capturar o "Slope" (inclinação) do comportamento do cliente, identificando reduções repentinas de consumo que precedem o estado de inadimplência.

* Tratamento de Variáveis de Alta Cardinalidade: Aplicação de técnicas de segmentação e clusterização em atributos de alta granularidade. Isso permitirá reduzir o ruído e melhorar a capacidade de generalização do modelo, transformando dados dispersos em agrupamentos com significado estatístico.

* Exploração de Dados Qualitativos: Aprofundamento na análise das tabelas dimensões para a extração de novos insights de negócio, permitindo que informações qualitativas sejam convertidas em atributos quantitativos (Feature Engineering).

* Avaliação de Ganho Preditivo (Lift): Realização de testes de importância de variáveis para mensurar o impacto incremental de cada novo atributo no poder preditivo do modelo (KS e AUC-ROC), garantindo que apenas as variáveis que agregam valor real sejam mantidas no pipeline final.